# Introduction to gen: design, track, and discover

Gen is a sequence version control system built around a graph database.
Rather than storing a sequence as a flat string, Gen breaks it into
nodes, each holding a short subsequence, connected by edges into a
graph. A path through that graph reconstructs a full sequence. When a
change is recorded, Gen adds new nodes and edges to represent the
altered subsequence and the new junctions around it. A path can then
take an alternative route through the new nodes, or skip nodes entirely
in the case of a deletion. Changes can be naturally occurring or
deliberately engineered. What makes this especially powerful is that the
graph does not just capture changes over time: it can represent the full
set of expected or observed variants across haplotypes, design libraries,
or a pangenome, all coexisting in one database with their differences
always queryable.

This notebook walks through a concrete lab scenario that shows where
that design pays off: you design an edit, apply it, confirm it with
search, export for sequencing, and then discover an unintended mutation
in the result.

## Setup

Import the package, then initialise a workspace and open a repository handle.

In [ ]:
import pathlib
import tempfile

import gen

# Locate fixture files bundled with this examples directory
EXAMPLES_DIR = pathlib.Path(__file__).parent if "__file__" in dir() else pathlib.Path(".").resolve()

# Create a fresh workspace directory and open a repository handle
WORK_DIR = pathlib.Path(tempfile.mkdtemp(prefix="gen-puc19-"))
repo = gen.Repository(str(WORK_DIR))
print(f"Repository at: {WORK_DIR}")

`Repository()` discovers the `.gen/` directory from the path argument (or the
current working directory when omitted) and opens the graph and operations
databases.  All subsequent operations go through this object.

## Import an annotated vector

We start with a fully annotated pUC19 sequence from NEB.  The GenBank
format carries feature annotations (genes, promoters, the MCS) that will
be preserved in the graph and displayed in the viewer.

In [ ]:
gbk_path = EXAMPLES_DIR / "puc19.gbk"
sample_wt = repo.import_genbank(str(gbk_path), sample="wt")

List the sequence graphs in the repository to confirm the import:

In [ ]:
sg_wt = sample_wt[0]
print(f"Sequence graph: {sg_wt.name} | sample: {sg_wt.sample_name}")

## Visualise the reference

Calling `.plot()` on a sequence graph returns an interactive widget that
renders directly in the notebook.  You can pan and zoom with the mouse.
The widget can also be controlled programmatically — useful for
reproducible navigation in a shared notebook.

In [ ]:
w_wt = sg_wt.plot()

# Navigate to the MCS — annotated features are first-class, so you can
# jump directly to any named feature rather than specifying coordinates.
annotations = sg_wt.list_annotations()
mcs = [
    a for a in annotations
    if any(k in a.name.lower() for k in ["mcs", "misc_feature", "multiple cloning"])
]
if mcs:
    w_wt.go_to(mcs[0])

w_wt

The widget renders a live snapshot of the sequence graph.  Annotations
from the GenBank file (lacZα, bla, the MCS) appear as coloured tracks
below the sequence.  `sg_wt.list_annotations()` returns all named features in
the database; passing one to `go_to()` centres the viewport on that
feature — no coordinate look-up required.

For a fully interactive, pannable, zoomable view from the command line,
run `gen view` in the same workspace.

## Design a Golden Gate conversion

The pUC19 multiple cloning site (MCS, positions 396–452) contains a set
of traditional type-II restriction sites.  We want to replace it with a
minimal Golden Gate entry site — two BsaI recognition sequences (GGTCTC)
flanking a stuffer, oriented to cut inward and expose compatible 4-nt
overhangs for assembly.

```
# Original MCS (1-based 396..452):
#   GAATTCGAGCTCGGTACCCGGGGATCCTCTAGAGTCGACCTGCAGGCATGCAAGCTT
#   EcoRI SacI  KpnI  SmaI  BamHI XbaI    SalI   PstI SphI HindIII
#
# Replacement: two BsaI sites flanking a stuffer, retaining EcoRI/HindIII ends.
#   GAATTC [BsaI→ GGTCTCA] [AATG overhang] [stuffer] [GCAT overhang] [←BsaI TGAGACC] AAGCTT
```

In [ ]:
golden_gate_mcs = "GAATTCGGTCTCAAATGCATCATCATCATGCATTGAGACCAAGCTT"

sample_gg = repo.update_with_sequence(
    golden_gate_mcs,
    sample="wt",
    new_sample="gg_design",
    # MCS coordinates from the GenBank misc_feature annotation (1-based 396..452),
    # converted to 0-based half-open interval for the region string.
    region_name="pUC19:395-452",
)

`update_with_sequence` creates a new sample (`"gg_design"`) that shares
the entire graph with `"wt"` except at the edited region, where the path
diverges through the new MCS nodes.

## Inspect the design

In [ ]:
sg_gg = sample_gg[0]

### Confirm the BsaI sites with search

Before synthesis, we verify the Golden Gate sites landed correctly by
searching for the BsaI recognition sequence on both strands.  `search()`
uses `sequence_kind="dna"` by default, which automatically searches
the reverse complement as well as the forward sequence.

In [ ]:
# GGTCTC is the BsaI recognition sequence (cuts 1 nt downstream on sense strand).
# DNA-mode search finds both the forward site and its reverse-complement counterpart.
hits = repo.search("GGTCTC", bgs=[sg_gg], sequence_kind="dna")
loci = hits[0][1]  # [(SequenceGraph, [Locus]), ...] — take the loci from the first match
print(f"BsaI sites found: {len(loci)}")

In [ ]:
w_gg = sg_gg.plot()
w_gg.go_to(loci[0])
w_gg

The widget centres on the first BsaI site, which sits at the boundary of
the new Golden Gate MCS.  Both paths are visible: the wild-type MCS and
the new Golden Gate site diverge at position 395 and reconverge 57 bp
later (original) or 46 bp later (new design).

Three hits.  To understand why, run the same search against the wild-type:

In [ ]:
hits_wt = repo.search("GGTCTC", bgs=[sg_wt], sequence_kind="dna")
wt_count = len(hits_wt[0][1]) if hits_wt else 0
print(f"BsaI sites in wild-type: {wt_count}")  # 1

The wild-type already carries one BsaI site — on the reverse strand at
position 1765, inside the **bla** coding sequence.  The design added
exactly two new sites (3 − 1 = 2), so the flanking Golden Gate sites are
present and correctly oriented.  The pre-existing site is the problem: if
left in place it will be cut alongside the intended assembly sites,
fragmenting the AmpR cassette.  It needs to be silently mutated out
before the construct goes to synthesis.

The `search()` method also accepts degenerate IUPAC codes, so you can
query by recognition sequence directly without pre-computing all
variants.  For example, EcoRI's sequence `GAATTC` is unambiguous, but a
site like HincII (`GTYRAC`, where Y = C/T and R = A/G) expands to four
possible hexamers — pass the IUPAC string and `search()` matches all of
them:

In [ ]:
hits_hincii = repo.search("GTYRAC", bgs=[sg_gg], sequence_kind="dna")
hincii_count = len(hits_hincii[0][1]) if hits_hincii else 0
print(f"HincII sites found: {hincii_count}")

For large genomes, consider calling `sg.build_index()` before
searching.  The index speeds up exact queries; degenerate IUPAC patterns
always use a full scan regardless.

### Navigate and highlight search results

Each element of the list returned by `search()` is a `Locus` with
`.start()` / `.end()` (`Position` objects) and `.slices`.  Pass a locus
directly to `go_to()` to centre the viewport on it, and to
`highlight_match()` to colour it on the canvas.

`go_to()` accepts `Position`, `Locus`, and `Annotation` objects — the
same call works for search results and annotation records.  Annotations
from one sample can be used to navigate a widget for a different sample
— this is safe and stable because annotations are stored as references
to graph nodes, not as linear sequence coordinates.  Nodes are shared
across samples in the same repository, so an annotation created for the
`"wt"` sample points to exactly the same graph nodes that appear in the
`"gg_design"` or `"sequenced"` graphs.

In [ ]:
# Jump to the first BsaI site and highlight both sites.
w_gg.go_to(loci[0])
w_gg.highlight_match(loci[0], color="cyan")
w_gg.highlight_match(loci[1], color="cyan")
w_gg

`clear_highlights()` removes all highlights without affecting the viewport:

In [ ]:
w_gg.clear_highlights()
w_gg

## Export and send for sequencing

In [ ]:
out_fa = WORK_DIR / "pUC19_gg_design.fa"
repo.export_fasta(str(out_fa), sample="gg_design")
print(f"Exported to {out_fa}")

You send `pUC19_gg_design.fa` to your synthesis provider.  A few days
later sequencing results arrive as a VCF.

## Import the sequencing result — and find a surprise

In [ ]:
vcf_path = EXAMPLES_DIR / "puc19_sequenced.vcf"

samples_seq = repo.update_with_vcf(
    str(vcf_path),
    reference="gg_design",
    sample="sequenced",
)
sample_seq = samples_seq[0]

In [ ]:
sg_seq = sample_seq[0]

In [ ]:
w_seq = sg_seq.plot()
w_seq

The graph now shows a branch at position 1904 — outside the MCS, inside
the **bla** (β-lactamase / AmpR) coding sequence.  The sequencing run
picked up a C→T substitution that was not in the design.  Because Gen
stores every sample as a path through the same graph, the mutation shows
up as a new node rather than a silent overwrite: the `gg_design` path
and the `sequenced` path diverge exactly here.

This is the kind of mutation that is easy to miss in a flat-file
workflow.  Gen surfaces it because "what changed between two samples" is
a native database query, not a diff of exported files.

## Version control on the command line

Everything above runs inside Python.  Gen also ships a command-line
interface that adds the operations that make it a true version control
system:

```sh
gen init
gen import --fasta reference.fa --sample wt
gen checkout -b gg_design
gen apply --vcf site.vcf
gen push origin gg_design
gen pull
gen log
```

`push`, `pull`, `checkout`, `log`, and `merge` work like their git
equivalents but operate on sequence graphs rather than text files.  A
Gen remote holds the full graph history: who edited which sample, when,
and how the paths diverged.

The Python bindings and the CLI share the same database, so you can
design a construct in a notebook, push it from the terminal, and a
colleague can pull the graph and visualise it in their own session —
with the full annotation and variant history intact.

To learn more about the CLI, run `gen --help` or visit the project
documentation.

## Further capabilities

The sections below cover additional API surface.  All examples are
self-contained and assume a repository initialised with
`gen init` (or the Python equivalent shown in Setup above).

### Additional import formats

#### GFA

GFA (Graphical Fragment Assembly) files encode sequence graphs directly.
Import a `.gfa` file as a new sample:

In [ ]:
# repo.import_gfa(str(FX / "assembly.gfa"), sample="assembly")

Export any sample back to GFA.  The optional `node_max` argument splits
nodes longer than the given length, useful for tools that expect bounded
node sizes:

In [ ]:
# repo.export_gfa("output.gfa", sample="assembly", node_max=1000)

#### Reference FASTA

`import_reference_fasta()` differs from `import_fasta()` in that it
marks the imported sample as a *reference* in the database.  This is the
right starting point when you plan to call variants relative to an
external aligner's reference: subsequent `update_with_vcf()` calls can
name this sample in the `reference` argument without ambiguity.

In [ ]:
# repo.import_reference_fasta("hg38.fa", reference="hg38")

#### GenBank updates and export

`update_with_genbank()` applies sequence edits and annotation changes
encoded in a GenBank file to an existing sample.  Set
`create_missing=True` to auto-create block groups for features not yet
in the database:

In [ ]:
# repo.update_with_genbank(
#     "corrected_annotation.gb",
#     sample="wt",
#     create_missing=True,
# )

Export any sample's annotations and sequence to GenBank format:

In [ ]:
# repo.export_genbank("pUC19_gg_design.gb", sample="gg_design")

### Graph export

Sequence graphs can be exported to popular Python graph libraries for
further analysis or visualisation.  Both NetworkX and RustworkX are
supported.

In [ ]:
try:
    import networkx as nx

    nx_graph = sg_gg.to_networkx()
    print(f"NetworkX DiGraph: {nx_graph.number_of_nodes()} nodes, {nx_graph.number_of_edges()} edges")
    print(f"Average degree: {sum(d for _, d in nx_graph.degree()) / nx_graph.number_of_nodes():.2f}")
except ImportError:
    print("NetworkX not installed. Run: pip install networkx")

In [ ]:
try:
    import rustworkx as rx
    import rustworkx.visualization as rxv

    rx_graph = sg_gg.to_rustworkx()
    print(f"RustworkX PyDiGraph: {rx_graph.num_nodes()} nodes, {rx_graph.num_edges()} edges")
    display(rxv.graphviz_draw(rx_graph))
except ImportError:
    print("RustworkX not installed. Run: pip install rustworkx")

### Combinatorial library design

`import_library()` builds a combinatorial block group from a list of
part columns.  Each column is a list of `SequencePart` objects where
each part has a name and a sequence.  Every combination of one part per
column becomes a path through the resulting graph.

The example below designs an expression cassette with three promoters,
three RBS variants, and two CDSes — 18 unique combinations:

In [ ]:
parts_list = [
    [gen.SequencePart("upstream", "AATTCGGATCCAAGCTT")],
    [
        gen.SequencePart("pTrc", "TTGACAATTAATCATCCGGCTCGTATAATGTGTGG"),
        gen.SequencePart("pT7",  "TAATACGACTCACTATA"),
        gen.SequencePart("pLac", "AATTGTGAGCGGATAACAATT"),
    ],
    [
        gen.SequencePart("rbs_strong", "AAAGAGGAGAAA"),
        gen.SequencePart("rbs_medium", "AAGAGGAG"),
        gen.SequencePart("rbs_weak",   "AGGAG"),
    ],
    [
        gen.SequencePart("gfp", "ATGAGTAAAGGAGAAGAACTTTTCACTGG"),
        gen.SequencePart("rfp", "ATGGCTTCCTCCGAAGACGTTATCAAAGAG"),
    ],
    [gen.SequencePart("terminator_T1", "GCGCAACGCAATTAATGTGAGTTAGCTCACTCATTAGGCACCCCAGGC")],
]

lib_repo = gen.Repository(str(WORK_DIR / "library"))
cassette_sg = lib_repo.import_library("expression-cassette", parts_list)

print(f"Paths in library graph: checking plot...")
cassette_sg.plot()

#### Applying a library update to an existing sample

`update_with_library()` works like `update_with_sequence()` but replaces
a region with a set of combinatorial variants.  The `path_name` argument
selects the region using the same `"<name>:<start>-<end>"` format as
other update functions:

In [ ]:
# repo.update_with_library(
#     sample="gg_design",
#     new_sample_name="gg_tagged",
#     path_name="pUC19:1200-1270",
#     parts_list=[
#         [gen.SequencePart("tag_a", "ATGATGATG"), gen.SequencePart("tag_b", "TGATGATGA")],
#     ],
# )

### Graph partitioning

#### Subgraphs

`derive_subgraph()` creates a new block group that is a coordinate slice
of an existing one.  The `region` argument uses
`"<name>:<start>-<end>"` (0-based half-open) or just `"<name>"` for the
full sequence:

In [ ]:
# repo.derive_subgraph(
#     sample="gg_design",
#     new_sample="mcs_region",
#     region="pUC19:390-460",
# )

Sequence graphs expose `subgraph()` as a shorthand that returns the new
sequence graph immediately:

In [ ]:
mcs_bg = sg_gg.subgraph("mcs_v2", 390, 460)
mcs_bg.plot()

#### Chunks and stitching

`sg.chunks()` splits a block group into contiguous fragments and returns
them as a list of sequence graphs.  Supply `chunk_size` for uniform splits
or `breakpoints` (a list of positions) for explicit cut points:

In [ ]:
chunks = sg_gg.chunks("gg_500bp_chunks", chunk_size=500)
print(f"Chunks created: {len(chunks)}")

`repo.stitch()` reverses the operation, concatenating a list of
sequence graph objects end-to-end into a new block group:

In [ ]:
reassembled = repo.stitch(
    bgs=chunks,
    new_sample="gg_reassembled",
    new_region="pUC19.reassembled",
)
print(f"Stitched: {reassembled.name} in sample '{reassembled.sample_name}'")
reassembled.plot()

### Annotation tracks

Any widget can display additional annotation panels below the sequence
graph.  `add_annotation_track()` is the unified entry point — supply
exactly one of `file`, `group`, or `annotations`:

- `file=` loads a GFF3 or BED file; `from_sample=` translates coordinates
  from the given sample's path space.
- `group=` loads an annotation group stored in the repository (created
  automatically on GenBank import).
- `annotations=` renders a list of `Annotation` objects you built manually.

In [ ]:
# w_gg.add_annotation_track(file="features.gff3", name="Features", from_sample="gg_design")
# w_gg

Annotation groups stored in the database (created automatically on
GenBank import) can be added by group name:

In [ ]:
# w_gg.add_annotation_track(group="GenBank annotations")
# w_gg

`clear_all_annotations()` removes all track panels without affecting the
viewport or highlights:

In [ ]:
# w_gg.clear_all_annotations()
# w_gg